# 

In [1]:
import litellm
import dotenv
dotenv.load_dotenv(".env")

def add(a: float, b: float) -> float:
    return a + b

def multiply(a: float, b: float) -> float:
    return a * b

def divide(a: float, b: float) -> float:
    if b == 0:
        return float('inf')
    return a / b

def subtract(a: float, b: float) -> float:
    return a - b

tool_functions = {
    "add": add,
    "multiply": multiply,
    "divide": divide,
    "subtract": subtract,
}


tools = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two numbers together",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"}
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "multiply",
            "description": "Multiply two numbers together",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "First number"},
                    "b": {"type": "number", "description": "Second number"}
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "divide",
            "description": "Divide the first number by the second. Returns infinity if dividing by zero.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Numerator"},
                    "b": {"type": "number", "description": "Denominator"}
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "subtract",
            "description": "Subtract the second number from the first",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "number", "description": "Number to subtract from"},
                    "b": {"type": "number", "description": "Number to subtract"}
                },
                "required": ["a", "b"],
            },
        },
    },
]

MODEL = "Qwen/Qwen3-VL-8B-Instruct-FP8"
BASE_URL = "http://localhost:8000/v1"
API_KEY = "your_api_key_here"
client = litellm.OpenAI( base_url=BASE_URL, api_key=API_KEY)

PROMPT_ARITHMETIC = """
You are a helpful assistant that can perform arithmetic operations using the provided tools.
Ensure to use tools for computations do not attempt to perform calculations yourself. You may use tools in sequence to arrive at the final answer how many ever steps needed.
You may be asked about out of scope topics, in which case you should respond that you can only perform arithmetic operations.
"""



# uv run vllm serve Qwen/Qwen3-VL-8B-Instruct-FP8   --limit-mm-per-prompt.video 0   --async-scheduling   --gpu-memory-utilization 0.6   --max-num-seqs 128 --enable-auto-tool-choice   --tool-call-parser hermes --max-model-len 3000

In [2]:
import json
def run_conversation(user_message: str, conversation_history=None):
    messages = [{"role":"system", "content": PROMPT_ARITHMETIC},{"role": "user", "content": user_message}]
    if conversation_history:
        messages = conversation_history 
    
    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )
        
        assistant_message = response.choices[0].message
        
        if assistant_message.tool_calls:
            print(f"Model wants to call {len(assistant_message.tool_calls)} tool(s)")
            
            for tool_call in assistant_message.tool_calls[:1]:  # Process one tool call at a time
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                print(f"  Calling {function_name}({function_args})")
                
                if function_name in tool_functions:
                    result = tool_functions[function_name](**function_args)
                else:
                    result = f"Error: Unknown function {function_name}"
                
                print(f"  Result: {result}")
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": f"{function_name}({function_args}) = {result}",
                })
        
        else:
            final_response = assistant_message.content
            messages.append({"role": "assistant", "content": final_response})
            print(f"\nFinal Answer: {final_response}")
            return final_response, messages

In [4]:
result, history = run_conversation("What is 135125 divided by 13 and add 15?")
print("Final Response:", result)

Model wants to call 2 tool(s)
  Calling divide({'a': 135125, 'b': 13})
  Result: 10394.23076923077
Model wants to call 1 tool(s)
  Calling add({'a': 10394.23076923077, 'b': 15})
  Result: 10409.23076923077

Final Answer: 10409.23076923077
Final Response: 10409.23076923077


In [5]:
result, history = run_conversation("What is the capital of France?")
print("Final Response:", result)


Final Answer: I can only perform arithmetic operations. The capital of France is Paris.
Final Response: I can only perform arithmetic operations. The capital of France is Paris.


In [6]:
history.append({"role": "user", "content": "What is the capital of France?"})
result, history = run_conversation("What is the capital of France?", conversation_history=history)
print("Final Response:", result)


Final Answer: The capital of France is Paris.
Final Response: The capital of France is Paris.


In [7]:
history.append({"role": "user", "content": "Скільки буде 25% від 1430?"})
result, history = run_conversation(user_message=None,conversation_history=history)
print("Final Response:", result)

Model wants to call 1 tool(s)
  Calling multiply({'a': 1430, 'b': 0.25})
  Result: 357.5

Final Answer: 25% від 1430 дорівнює 357.5.
Final Response: 25% від 1430 дорівнює 357.5.


In [8]:
history.append({"role": "user", "content": "Скільки буде 1430% від 25?"})
result, history = run_conversation(user_message=None,conversation_history=history)
print("Final Response:", result)

Model wants to call 1 tool(s)
  Calling multiply({'a': 25, 'b': 14.3})
  Result: 357.5

Final Answer: 1430% від 25 дорівнює 357.5.
Final Response: 1430% від 25 дорівнює 357.5.


In [9]:
history.append({"role": "user", "content": "У Олени є 2.5 × 8 + 14.75 + 6 × 9.2 груш. Вона віддала своїм сусідам 18.4 груші. Скільки груш залишилось у Олени?"})
result, history = run_conversation(user_message=None,conversation_history=history)
print("Final Response:", result)

Model wants to call 2 tool(s)
  Calling multiply({'a': 2.5, 'b': 8})
  Result: 20.0
Model wants to call 1 tool(s)
  Calling add({'a': 20.0, 'b': 14.75})
  Result: 34.75
Model wants to call 1 tool(s)
  Calling multiply({'a': 6, 'b': 9.2})
  Result: 55.199999999999996

Final Answer: Спочатку обчислимо загальну кількість груш у Олени:

1. \( 2.5 \times 8 = 20.0 \)
2. \( 20.0 + 14.75 = 34.75 \)
3. \( 6 \times 9.2 = 55.2 \)
4. \( 34.75 + 55.2 = 89.95 \)

Тепер віднімемо кількість груш, які вона віддала сусідам:

\( 89.95 - 18.4 = 71.55 \)

Отже, у Олени залишилось **71.55 груш**.
Final Response: Спочатку обчислимо загальну кількість груш у Олени:

1. \( 2.5 \times 8 = 20.0 \)
2. \( 20.0 + 14.75 = 34.75 \)
3. \( 6 \times 9.2 = 55.2 \)
4. \( 34.75 + 55.2 = 89.95 \)

Тепер віднімемо кількість груш, які вона віддала сусідам:

\( 89.95 - 18.4 = 71.55 \)

Отже, у Олени залишилось **71.55 груш**.


# 2. Comparative SDK Analysis 

For this part I decided to move with autogen instead of langchain/langraph (since there is little to no differences and I do not like langchain) and Openai SDK. 

In [10]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
)


async def get_weather(city: str) -> str:
    """Get the weather for a given city."""
    return f"The weather in {city} is 73 degrees and Sunny."

async def get_stock(ticker: str) -> str:
    """Get the stock price for a given ticker symbol."""
    return f"The stock price of {ticker} is $150.00."

async def send_email(topic: str, receiver: str, body: str) -> str:
    """Send an email on a given topic."""
    return f"Email on '{topic}' has been sent successfully."

agent = AssistantAgent(
    name="weather_agent",
    model_client=model_client,
    tools=[get_weather, get_stock, send_email],
    system_message="You are a helpful assistant.",
    reflect_on_tool_use=True,
)


In [11]:
def print_history(res):
    if isinstance(res, list):
        for message in res:
            role = getattr(message, 'source', 'unk')
            content = getattr(message, 'content', '')
            print(f"{role}: {content}\n")
        return

    for message in res.messages:
        role = getattr(message, 'source', 'unk')
        content = getattr(message, 'content', '')
        print(f"{role}: {content}\n")

In [12]:
res = await agent.run(task="What's the price of S&P 500 today?")
messages = await agent._model_context.get_messages()
print_history(messages)

user: What's the price of S&P 500 today?

weather_agent: [FunctionCall(id='call_INptY8uGX4c4wmeSWNqCUEsg', arguments='{"ticker":"SPY"}', name='get_stock')]

unk: [FunctionExecutionResult(content='The stock price of SPY is $150.00.', name='get_stock', call_id='call_INptY8uGX4c4wmeSWNqCUEsg', is_error=False)]

weather_agent: The current price of the S&P 500 ETF (SPY) is $150.00.



In [13]:
res = await agent.run(task="Send email about this")
messages = await agent._model_context.get_messages()
print_history(messages)

user: What's the price of S&P 500 today?

weather_agent: [FunctionCall(id='call_INptY8uGX4c4wmeSWNqCUEsg', arguments='{"ticker":"SPY"}', name='get_stock')]

unk: [FunctionExecutionResult(content='The stock price of SPY is $150.00.', name='get_stock', call_id='call_INptY8uGX4c4wmeSWNqCUEsg', is_error=False)]

weather_agent: The current price of the S&P 500 ETF (SPY) is $150.00.

user: Send email about this

weather_agent: To whom would you like to send the email and what should the subject of the email be?



In [14]:
res = await agent.run(task="send to nazar@gmail.com, write about today's weather in San Francisco and the stock price of S&P 500, do not confirm with me")
messages = await agent._model_context.get_messages()
print_history(messages)

user: What's the price of S&P 500 today?

weather_agent: [FunctionCall(id='call_INptY8uGX4c4wmeSWNqCUEsg', arguments='{"ticker":"SPY"}', name='get_stock')]

unk: [FunctionExecutionResult(content='The stock price of SPY is $150.00.', name='get_stock', call_id='call_INptY8uGX4c4wmeSWNqCUEsg', is_error=False)]

weather_agent: The current price of the S&P 500 ETF (SPY) is $150.00.

user: Send email about this

weather_agent: To whom would you like to send the email and what should the subject of the email be?

user: send to nazar@gmail.com, write about today's weather in San Francisco and the stock price of S&P 500, do not confirm with me

weather_agent: [FunctionCall(id='call_WYTT782VMn7PLMsu4sCstIUb', arguments='{"city": "San Francisco"}', name='get_weather'), FunctionCall(id='call_LGYbXHslSrRNfY4P2zmneP4L', arguments='{"ticker": "SPY"}', name='get_stock')]

unk: [FunctionExecutionResult(content='The weather in San Francisco is 73 degrees and Sunny.', name='get_weather', call_id='call_WY

In [38]:
import json
from openai import OpenAI

client = OpenAI()

def get_weather(city: str) -> str:
    return f"Weather in {city}: Sunny, 72°F"

def get_stock(ticker: str) -> str:
    return f"{ticker}: $150.25"

def send_email(topic: str, receiver: str, body: str) -> str:
    return f"Email sent to {receiver}"

available_functions = {
    "get_weather": get_weather,
    "get_stock": get_stock,
    "send_email": send_email,
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "The city to get weather for"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_stock",
            "description": "Get the stock price for a given ticker symbol.",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {"type": "string", "description": "The stock ticker symbol"}
                },
                "required": ["ticker"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "send_email",
            "description": "Send an email on a given topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "topic": {"type": "string", "description": "The email topic/subject"},
                    "receiver": {"type": "string", "description": "The email recipient"},
                    "body": {"type": "string", "description": "The email body content"}
                },
                "required": ["topic", "receiver", "body"]
            }
        }
    }
]

def run_agent(user_message: str = "", history = None):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": user_message}
    ]
    
    if history:
        messages = history

    while True:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        assistant_message = response.choices[0].message

        if assistant_message.tool_calls:
            messages.append({
                "role": "assistant",
                "content": assistant_message.content,
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments
                        }
                    } for tc in assistant_message.tool_calls
                ]
            })
        else:
            messages.append({
                "role": "assistant",
                "content": assistant_message.content
            })
        
        if assistant_message.tool_calls:
            for tool_call in assistant_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                function_to_call = available_functions[function_name]
                result = function_to_call(**function_args)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })
            continue
        
        return assistant_message.content, messages

In [39]:
response, history = run_agent("What's the weather in San Francisco and the stock price of AAPL?")
print(history)

[{'role': 'system', 'content': 'You are a helpful assistant.'}, {'role': 'user', 'content': "What's the weather in San Francisco and the stock price of AAPL?"}, {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'call_mmhihLsp8uuLbYQl50f9vrJB', 'type': 'function', 'function': {'name': 'get_weather', 'arguments': '{"city": "San Francisco"}'}}, {'id': 'call_9CueOZRbWHRtK7vpJRj0BUxP', 'type': 'function', 'function': {'name': 'get_stock', 'arguments': '{"ticker": "AAPL"}'}}]}, {'role': 'tool', 'tool_call_id': 'call_mmhihLsp8uuLbYQl50f9vrJB', 'content': 'Weather in San Francisco: Sunny, 72°F'}, {'role': 'tool', 'tool_call_id': 'call_9CueOZRbWHRtK7vpJRj0BUxP', 'content': 'AAPL: $150.25'}, {'role': 'assistant', 'content': 'The weather in San Francisco is sunny with a temperature of 72°F. The current stock price of AAPL is $150.25.'}]


In [40]:
history

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user',
  'content': "What's the weather in San Francisco and the stock price of AAPL?"},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'call_mmhihLsp8uuLbYQl50f9vrJB',
    'type': 'function',
    'function': {'name': 'get_weather',
     'arguments': '{"city": "San Francisco"}'}},
   {'id': 'call_9CueOZRbWHRtK7vpJRj0BUxP',
    'type': 'function',
    'function': {'name': 'get_stock', 'arguments': '{"ticker": "AAPL"}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_mmhihLsp8uuLbYQl50f9vrJB',
  'content': 'Weather in San Francisco: Sunny, 72°F'},
 {'role': 'tool',
  'tool_call_id': 'call_9CueOZRbWHRtK7vpJRj0BUxP',
  'content': 'AAPL: $150.25'},
 {'role': 'assistant',
  'content': 'The weather in San Francisco is sunny with a temperature of 72°F. The current stock price of AAPL is $150.25.'}]

In [42]:
history.append({"role": "user", "content": "Can you send an email about this to john@example.com?"})
response, history = run_agent(history=history)
history

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user',
  'content': "What's the weather in San Francisco and the stock price of AAPL?"},
 {'role': 'assistant',
  'content': None,
  'tool_calls': [{'id': 'call_mmhihLsp8uuLbYQl50f9vrJB',
    'type': 'function',
    'function': {'name': 'get_weather',
     'arguments': '{"city": "San Francisco"}'}},
   {'id': 'call_9CueOZRbWHRtK7vpJRj0BUxP',
    'type': 'function',
    'function': {'name': 'get_stock', 'arguments': '{"ticker": "AAPL"}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_mmhihLsp8uuLbYQl50f9vrJB',
  'content': 'Weather in San Francisco: Sunny, 72°F'},
 {'role': 'tool',
  'tool_call_id': 'call_9CueOZRbWHRtK7vpJRj0BUxP',
  'content': 'AAPL: $150.25'},
 {'role': 'assistant',
  'content': 'The weather in San Francisco is sunny with a temperature of 72°F. The current stock price of AAPL is $150.25.'},
 {'role': 'user',
  'content': 'Can you send an email about this to john@example.com?'},
 {'role': 'assist

Generally speaking, they are quite the same, however with OpenAI SDK we have to work on a lower level of abstraction and we actually have to execute functions instead of just passing them to some wrapper. We have to manage history and so on.

With AutoGen, you simply define Python functions and pass them to the agent. The framework handles JSON schema conversion, tool execution, and conversation flow automatically.

With OpenAI SDK, you must manually define JSON schemas, implement the tool execution loop, append results to history, and handle edge cases. The history management is critical - every assistant message with tool_calls must be followed by matching tool responses, or the API will reject the request.
AutoGen is better for rapid prototyping. OpenAI SDK is preferable when you need full control or want to understand the underlying mechanics.

# 3. Multi-Agent System (MAS)

In [47]:

import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.teams import SelectorGroupChat
from autogen_ext.models.openai import OpenAIChatCompletionClient
from ddgs import DDGS

model_client = OpenAIChatCompletionClient(
    model="gpt-4o-mini",
)

def search_web_tool(query: str) -> str:
    results = DDGS().text(query, max_results=5)
    return results

researcher = AssistantAgent(
    name="Researcher",
    description="Searches the web to find current facts, statistics, and sources. Analyzes and summarizes findings. Call this agent when you need to look up information or verify claims.",
    system_message="""You are a researcher. Provide factual, well-analyzed responses.
    You have access to a web search tool to gather up-to-date information.""",
    model_client=model_client,
    tools=[search_web_tool],
    reflect_on_tool_use=True,
)

writer = AssistantAgent(
    name="Writer",
    description="Transforms raw information into polished content. Writes articles, reports, and summaries. Call this agent after research is complete to create the final output.",
    system_message="You are a writer. Create engaging, polished content. After finishing your writing, mention 'TERMINATE' to end the conversation.",
    model_client=model_client,
)

selector_prompt = """Select an agent to perform task.

{roles}

Current conversation context:
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Only select one agent.
"""


text_mention_termination = TextMentionTermination("TERMINATE")
max_messages_termination = MaxMessageTermination(max_messages=5)
termination = text_mention_termination | max_messages_termination

team = SelectorGroupChat(
    [researcher, writer],
    name="Supervisor",
    model_client=model_client,
    termination_condition=termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True, 
)

SelectorGroupChat is a "supervisor" it selects agents and gives them subtasks, agents - researcher, writer to do them, the final agent is the writer, it may end completition with "TERMINATE" word which indicates end of the flow, another option to end flow is surpassing limit of agent calls - 5.

In [ ]:
res = await team.run(task="Що то за Україньска ЛЛМ така Лапа?")


In [ ]:
print_history(res)

user: Що то за Україньска ЛЛМ така Лапа?

Researcher: [FunctionCall(id='call_3sx1wun6fN53XUPCz269je79', arguments='{"query":"ЛЛМ Лапа українська"}', name='search_web_tool')]

Researcher: [FunctionExecutionResult(content='[{\'title\': \'Лапа ЛЛМ: Найшвидша модель українського NLP! #shorts\', \'href\': \'https://www.youtube.com/watch?v=nFWyJHFVVRM\', \'body\': \'Нова Лапа ЛЛМ оптимізована для української мови, зменшує обчислювальні витрати в 1.5 рази. Найшвидша модель українського NLP! Переклад, резюмування, та аналі...\'}, {\'title\': \'Відкрита українська мовна модель Lapa LLM отримала публічний реліз\', \'href\': \'https://dev.ua/news/vidkryta-ukrainska-movna-model-lapa-llm-otrymala-publichnyi-reliz-1761343981\', \'body\': \'Команда українських дослідників презентувала Lapa LLM v0.1.2 — велику мовну модель на базі Gemma-3-12B, оптимізовану для роботи з українською мовою. Завдяки новому токенізатору Lapa LLM замінила 80 000 токенів із 250 000 на українські, зберігши якість оригінальної

below we can see that selector always decided to call researcher which basically did not let writer to "finish" flow, this can be fixed by refining description of agents and refining prompt of the selector, though without specific aim it is quite hard to refine it, so this is one case when it fails.

In [57]:
res = await team.run(task="А хто її розробив і для чого?")
print_history(res)

user: А хто її розробив і для чого?

Researcher: Lapa LLM була розроблена командою українських дослідників з Українського католицького університету, Київського політехнічного інституту, Львівської політехніки та інституту AGH в Кракові. Основною метою розробки цієї моделі є створення потужного інструменту для обробки природної мови українською мовою.

### Основні цілі розробки Lapa LLM:
1. **Покращення якості обробки українського тексту**: Модель спеціально адаптована для врахування особливостей української мови, що дозволяє досягти більш високої точності в генерації та аналізі тексту.

2. **Сприяння розвитку штучного інтелекту в Україні**: Lapa LLM має на меті посилити наукові дослідження у сфері штучного інтелекту, підтримуючи місцевих розробників та науковців.

3. **Збереження та популяризація української мови**: Завдяки Lapa LLM, українська мова отримує нові інструменти для використання в різних технологічних сферах, що може сприяти її поширенню в цифровому середовищі.

4. **Виріше